# 04 — Phase 3: GQA + KIVI (2-bit KV Cache)

**技术**: KIVI — 2-bit 非对称量化 KV Cache  
**框架**: PyTorch 原生 + KIVI CUDA Backend（`kivi_gemv`）

> ⚠️ 本 notebook 使用 **PyTorch 原生推理路径**（`metrics.py`），不经过 vLLM。
> KIVI 和 vLLM 的 PagedAttention 在同一层（KV Cache 管理层）做完全不同的事情，
> 两者无法并行运行。这是控制变量法的第三组：只改 **数值精度** 这一个变量。

### KIVI 原理
- **K Cache**: per-channel 量化（沿 head_dim 方向统计 min/max）
- **V Cache**: per-token 量化（沿 seq_len 方向统计 min/max）
- **Residual window**: 最近 128 个 token 保持 FP16（保证质量）
- 理论压缩 **~5-8×**

### CUDA Backend
KIVI 的自定义 CUDA kernel（`quant/csrc/gemv_cuda.cu`）提供：
- **`bgemv2_kernel_outer_dim`**: 2-bit packed GEMV（Q × K_quant, attn × V_quant）
- 每个 warp 读 32 个 packed uint32，在线反量化 `dequant = scale * (int & 0x3) + zero`
- 支持 GQA：通过 `nh / nh_kv` ratio 自动处理 KV head 复用

编译方法（在 l4t-pytorch 容器内）：
```bash
cd ~/KIVI && pip install -e .
cd quant && TORCH_CUDA_ARCH_LIST="8.7" pip install -e .
python -c "import kivi_gemv; print('OK')"
```

### 预期效果
- KV Cache 显存占用从 FP16 降低 5-8×
- **TPOT 降低**: decode 每步搬运的 KV 数据量减少（显存带宽是 Jetson 瓶颈）
- **TTFT 基本不变**: prefill 阶段 KV 尚未量化
- **PPL 可能略升**: 2-bit 有损压缩 → 需要检查质量退化

In [1]:
import sys, gc, torch
sys.path.insert(0, '..')

from src.metrics import measure_generation, run_benchmark, find_oom_threshold, print_memory_budget, JETSON_USABLE_GB
from src.perplexity import compute_ppl_with_kv_cache
from src.dataset_utils import load_prompts
from src.jetson_utils import load_model_safe, aggressive_cleanup

# 检查底层算子是否就绪
import kivi_gemv
print(f"✅ KIVI C++ 算子已加载: {dir(kivi_gemv)}")
import os
# 必须在导入 transformers 之前执行！取消不稳定的 Rust 加速器
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"


✅ KIVI C++ 算子已加载: ['__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'gemv_forward_cuda', 'gemv_forward_cuda_outer_dim']


In [2]:
import torch
from transformers import AutoTokenizer
# ... 其他各种 import ...

MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"

print("开始使用原生模式稳定下载...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model, tokenizer = load_model_safe(MODEL_NAME, device="cuda")





/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/venv/lib/python3.12/site-packages/transformers/utils/hub.py:105: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


开始使用原生模式稳定下载...


Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:10<00:00,  5.02s/it]


✓ Loaded unsloth/Llama-3.2-3B-Instruct (torch.float16)  GPU mem: 5.98 GB


In [3]:
import kivi_gemv
print("KIVI C++ 引擎内部接口:")
for func in dir(kivi_gemv):
    if not func.startswith('__'):
        print(f" - {func}")

KIVI C++ 引擎内部接口:
 - gemv_forward_cuda
 - gemv_forward_cuda_outer_dim


In [6]:
from src.qwen_kivi_2 import patch_llama_with_kivi, KIVICUDACache

BITS = 2  # C++ 底层死规定，只能填 2
GROUP_SIZE = 32
RESIDUAL_LENGTH = 128

kivi_config = {
    'k_bits': BITS, 'v_bits': BITS, 
    'group_size': GROUP_SIZE, 'residual_length': RESIDUAL_LENGTH
}

# patch进原有的模型中
model = patch_llama_with_kivi(model, kivi_config)

✅ Llama Patched with Modern KIVI! layers=28, bits=2, residual=128


In [7]:
import pandas as pd  # <--- 加上这一行导入 pandas
prompts = load_prompts('../results/ehr_prompts_llama3.2.json')

# Benchmark 测速 (传入我们定制的 KIVICUDACache)
results_kivi = run_benchmark(
    model, tokenizer, prompts,
    cache_factory=KIVICUDACache,  
    max_new_tokens=256,
    warmup_runs=2,
)

df = pd.DataFrame(results_kivi)
print(f"\n⚡ 平均 TPOT: {df['tpot_ms'].mean():.1f} ms")

# PPL 测试 (调用你修改后的自回归 perplexity 函数)
eval_texts = [p['reference_answer'] for p in prompts[:20] if p.get('reference_answer')]

ppl_result = compute_ppl_with_kv_cache(
    model, tokenizer, eval_texts,
    cache_factory=KIVICUDACache,
    max_length=512
)

print(f"🔥 KIVI 2-bit 真实 PPL: {ppl_result['ppl']:.2f}")

Running 2 warmup cycles...


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


Warmup complete. Starting benchmark...

  [1/50] ttft=1651ms  tpot=231.8ms  peak=6326MB  out=256tok


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


  [2/50] ttft=1852ms  tpot=230.2ms  peak=6326MB  out=147tok


...

  [42/50] ttft=1669ms  tpot=230.4ms  peak=6315MB  out=256tok
  [43/50] ttft=1641ms  tpot=231.2ms  peak=6321MB  out=256tok
  [44/50] ttft=1630ms  tpot=231.2ms  peak=6320MB  out=251tok
  [45/50] ttft=1630ms  tpot=231.4ms  peak=6321MB  out=256tok
  [46/50] ttft=1634ms  tpot=231.2ms  peak=6325MB  out=191tok
  [47/50] ttft=1640ms  tpot=231.2ms  peak=6324MB  out=256tok
  [48/50] ttft=1635ms  tpot=231.3ms  peak=6324MB  out=256tok
  [49/50] ttft=1626ms  tpot=231.6ms  peak=6319MB  out=256tok


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


  [50/50] ttft=1840ms  tpot=231.5ms  peak=6318MB  out=256tok

⚡ 平均 TPOT: 231.0 ms


KIVI Decode PPL:   0%|                                                                                                                                 | 0/20 [00:00<?, ?it/s]/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
KIVI Decode PPL: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:27<00:00,  1.37s/it]

🔥 KIVI 2-bit 真实 PPL: 20.68


In [8]:
import pandas as pd

df = pd.DataFrame(results_kivi)
df['config'] = 'GQA+KIVI'
df.to_csv('../results/gqa_kivi_llama3.2.csv', index=False)

print(df[['ttft_ms', 'tpot_ms', 'peak_memory_mb',
           'kv_cache_memory_mb', 'memory_utilization']].describe().round(1))

       ttft_ms  tpot_ms  peak_memory_mb  kv_cache_memory_mb  \
count     50.0     50.0            50.0                50.0   
mean    1747.4    231.0          6321.5               181.7   
std      111.9      0.6             5.1                 5.1   
min     1620.0    229.6          6315.1               175.4   
25%     1633.3    230.6          6318.6               178.8   
50%     1822.1    231.0          6321.3               181.4   
75%     1834.5    231.3          6323.6               183.8   
max     2075.3    232.2          6350.5               210.7   

       memory_utilization  
count                50.0  
mean                  0.6  
std                   0.0  
min                   0.6  
25%                   0.6  
50%                   0.6  
75%                   0.6  
max                   0.6  


---
## PPL 退化评估

2-bit 量化可能导致生成质量下降。PPL 增加 > 5% 视为显著退化。

In [9]:
# 使用我们重写的逐字解码版 PPL
eval_texts = [p['reference_answer'] for p in prompts[:20] if p.get('reference_answer')]

ppl_result = compute_ppl_with_kv_cache(
    model, tokenizer, eval_texts,
    cache_factory=None, # patch 模型自管理
    max_length=512
)

print(f"🔥 KIVI 2-bit 真实解压 PPL: {ppl_result['ppl']:.2f}")

KIVI Decode PPL:   0%|                                                                                                                                 | 0/20 [00:00<?, ?it/s]/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
KIVI Decode PPL: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:27<00:00,  1.36s/it]

🔥 KIVI 2-bit 真实解压 PPL: 19575.68


In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
print("🧹 显存已释放，准备下一组实验。")

In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / (1024**2):.0f} MB")